# Enhance Optics with Database Chat

### Setup

In [19]:
import pandas as pd
import numpy as np
import pickle
from dotenv import load_dotenv
import os
from typing import List
import openai
import redis
from redis.commands.search.indexDefinition import (
    IndexDefinition,
    IndexType
)
from redis.commands.search.query import Query
from redis.commands.search.field import (
    TextField,
    VectorField
)
# from utils import ingest_query

In [20]:
load_dotenv()
open_api_key = os.getenv("OPENAI_API_KEY")
openai_client = openai.OpenAI(api_key=open_api_key)

In [21]:
REDIS_HOST =  "localhost"
REDIS_PORT = 6379
REDIS_PASSWORD = "" # default for passwordless Redis

# Connect to Redis
redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD
)
redis_client.ping()

True

In [22]:
EMBEDDING_MODEL = "text-embedding-3-small"

In [23]:

# --- Query Tools ---
# Helper function to search on multiple fields
def create_hybrid_field(field_name: str, value: str) -> str:
    return f'@{field_name}:"{value}"'

# Search top K results from Redis using Embeddings
def search_redis(
    openai_client: openai.OpenAI,
    redis_client: redis.Redis,
    user_query: str,
    index_name: str = "embeddings-index",
    vector_field: str = "content_vector",
    return_fields: list = ["title", "text", "chunk_id", "vector_score"],
    hybrid_fields = "*",
    k: int = 5,
) -> List[dict]:
    # Creates embedding vector from user query
    embedded_query = openai_client.embeddings.create(input=user_query,
                                            model=EMBEDDING_MODEL,
                                            ).data[0].embedding
    # Prepare the Query
    base_query = f'{hybrid_fields}=>[KNN {k} @{vector_field} $vector AS vector_score]'
    query = (
        Query(base_query)
         .return_fields(*return_fields)
         .sort_by("vector_score")
         .paging(0, k)
         .dialect(2)
    )
    params_dict = {"vector": np.array(embedded_query).astype(dtype=np.float32).tobytes()}

    # Perform vector search
    results = redis_client.ft(index_name).search(query, params_dict)
    for i, article in enumerate(results.docs):
        score = 1 - float(article.vector_score)
        # print(f"{i}. {article.title}\n(Score: {round(score ,3) })")
    return results.docs

def add_context(
    chunk_ids: list,
    file_name: str,
    title_dict: dict,
    chunk_dict: dict,
):
    with open(f'data/prompt.txt', 'r', encoding='utf-8') as f:
        prompt = f.read()

    for i, chunk_id in enumerate(chunk_ids):
        video_id = chunk_id.split('_videoid:')[1].split('_chunk:')[0]
        prompt = f"{prompt}### Context Document {i}\nTitle: \t{title_dict[video_id]}\nContext: \t{chunk_dict[chunk_id]}\n\n\n"

    os.makedirs('data/prompts', exist_ok=True)
    with open(f"data/prompts/{file_name}.txt", "w") as f:
        f.write(prompt)

def ingest_query(
    openai_client: openai.OpenAI,
    redis_client: redis.Redis,
    user_query: str,
    k: int = 5,
    vector_field: str = "content_vector",
):
    results = search_redis(
        openai_client, 
        redis_client, 
        user_query, 
        vector_field='content_vector', 
        k=10
    )

    chunk_ids = [x.chunk_id for x in results]

    with open('data/title_dict.pkl', 'rb') as f:
        title_dict = pickle.load(f)
    with open('data/chunk_dict.pkl', 'rb') as f:
        chunk_dict = pickle.load(f)

    add_context(
        chunk_ids,
        'prompt_0',
        title_dict,
        chunk_dict
    )



In [24]:
user_query = "How do I become more productive?"

ingest_query(openai_client, redis_client, user_query)

with open("data/prompts/prompt_0.txt", "r", encoding="utf-8") as f:
    instructions = f.read()

response = openai_client.responses.create(
    model="o4-mini",
    instructions=instructions,
    input=user_query,
)

In [27]:
response.output_text

'Becoming more productive is best achieved by layering several independent, science-backed approaches. Below are five unique strategies, ordered roughly by their relative impact:\n\n1. Optimize Your Focus Architecture (High Impact)  \n  • Use Ultradian Cycles: Work in 90–120-minute “deep focus” blocks, then take a 15–30-minute break.  \n  • External Cues: Employ a simple metronome or timer to signal start/end of focus periods.  \n  • Maximal Density of Effort: Within each block, minimize multitasking—aim for maximum repetitions of your core task.\n\n2. Manage Dopamine & Reward Scheduling (High Impact)  \n  • Intermittent Reinforcement: Break big goals into subgoals, but only celebrate some (not all) to keep dopamine circuits primed.  \n  • Occasional “Blunting”: After major milestones, deliberately under-celebrate to prevent reward crashes.  \n  • Variable Rewards: Randomize small treats (e.g., 5-minute walk, a favorite snack) so they remain motivating.\n\n3. Design Your Environment (M

In [28]:
with open(f"response.txt", "w") as f:
    f.write(response.output_text)